# Opdracht 4 – ETL Implementatie Data Warehouse
## Robbert & Mees

In deze opdracht implementeren wij de ETL-processen van het Data Warehouse. 
We laden data vanuit het Source Data Model (SDM) naar het DWH en passen 
Slowly Changing Dimensions Type 1 en Type 2 toe.

## Inlaadstrategie

In deze opdracht maken wij gebruik van de **Incremental Data Loading** strategie.

Reden:
- we willen alleen nieuwe en gewijzigde data verwerken
- nodig voor SCD Type 2 (historie behouden)


Setup & Connecties (NOG INVULLEN)

In [ ]:
import pandas as pd
import sqlite3
from loguru import logger
from datetime import datetime # nodig voor dimensies (begin/eindtijd)

sdm_conn = sqlite3.connect('../Source Data Model/BikeToDrive_V3_OLTP.db', timeout=30)
dwh_conn = sqlite3.connect('../Data Warehouse/BikeToDrive_DWH_V3.db', timeout=30)
sdm_cursor = sdm_conn.cursor()
dwh_cursor = dwh_conn.cursor()

logger.info("SDM en DWH connecties succesvol.")

Dim_Klant (SCD TYPE 2) ROBBERT

In [ ]:
# ETL Dim_Klant - SCD Type 2
# Doel:
# We vullen Dim_Klant in het DWH met SCD Type 2 logica.
#
#
# SCD Type 2-regels:
# - record bestaat nog niet in DWH -> INSERT
# - record bestaat wel, maar attribuutwaarden zijn gewijzigd
#     -> oude versie afsluiten + nieuwe versie INSERT
# - record bestaat wel en is ongewijzigd -> niets doen
#
# Business key  = (klantnr, source_id)
# Surrogate key = klant_sk

logger.info("Start ETL Dim_Klant (SCD Type 2 met business key klantnr + source_id)")

# 1. Brondata ophalen
# We lezen klantdata uit beide bron-tabellen.
# Bij het inlezen voegen we direct een source_id toe,
# zodat elk record eenduidig herleidbaar is naar de bron.

query_accessoire_klant = """
SELECT
    klantnr,
    naam,
    adres,
    woonplaats,
    geslacht,
    geboortedatum
FROM Accessoire_Verkoop_Klant
"""

query_fiets_klant = """
SELECT
    klantnr,
    naam,
    adres,
    woonplaats,
    geslacht,
    geboortedatum
FROM Fiets_Verkoop_Klant
"""

df_accessoire_klant = pd.read_sql_query(query_accessoire_klant, sdm_conn)
df_fiets_klant = pd.read_sql_query(query_fiets_klant, sdm_conn)

# Voeg source_id toe
df_accessoire_klant["source_id"] = "Accessoire_Verkoop_Klant"
df_fiets_klant["source_id"] = "Fiets_Verkoop_Klant"

logger.info(f"Aantal klant-rijen uit accessoirebron: {len(df_accessoire_klant)}")
logger.info(f"Aantal klant-rijen uit fietsbron: {len(df_fiets_klant)}")


# 2. Brondata samenvoegen
# Omdat source_id nu onderdeel is van de business key,
# mogen dezelfde klantnummers uit verschillende bronnen
# gewoon naast elkaar bestaan.
# Daarom vervalt de oude conflictlogica volledig.
# We zetten de twee bronnen simpelweg onder elkaar.

df_klant_source = pd.concat(
    [df_accessoire_klant, df_fiets_klant],
    ignore_index=True
)

logger.info(f"Aantal records in samengestelde bronset: {len(df_klant_source)}")

# Controle op dubbelen binnen dezelfde business key
# (alleen handig als je wilt valideren dat dezelfde combinatie
# klantnr + source_id niet per ongeluk dubbel in de bron zit)
# dit komt echter niet voor in onze case, maar is misschien handig als er wel dubbele records binnenkomen
duplicate_bk = df_klant_source.duplicated(subset=["klantnr", "source_id"], keep=False)

if duplicate_bk.any():
    logger.warning("Er zijn dubbele records gevonden binnen dezelfde business key (klantnr + source_id).")
    logger.warning(f"Aantal dubbele bronrecords: {duplicate_bk.sum()}")


# 3. Actuele records uit het DWH ophalen
# Voor SCD Type 2 vergelijken we alleen met de actieve versie.
# Daarom halen we alleen records op met eindtijd IS NULL.

query_dwh_klant = """
SELECT
    klant_sk,
    klantnr,
    source_id,
    naam,
    adres,
    woonplaats,
    geslacht,
    geboortedatum,
    begintijd,
    eindtijd
FROM Dim_Klant
WHERE eindtijd IS NULL
"""

df_klant_dwh = pd.read_sql_query(query_dwh_klant, dwh_conn)

logger.info(f"Aantal actuele klanten in DWH: {len(df_klant_dwh)}")


# 4. Helperfunctie: is de klant veranderd?
# We vergelijken alleen de inhoudelijke attributen.
# De business key zelf (klantnr + source_id) gebruiken we
# om het juiste actuele record te vinden.

def klant_is_changed(source_row, dwh_row):
    return (
        str(source_row["naam"]) != str(dwh_row["naam"]) or
        str(source_row["adres"]) != str(dwh_row["adres"]) or
        str(source_row["woonplaats"]) != str(dwh_row["woonplaats"]) or
        str(source_row["geslacht"]) != str(dwh_row["geslacht"]) or
        str(source_row["geboortedatum"]) != str(dwh_row["geboortedatum"])
    )

# 5. Vooraf bepalen hoeveel records nieuw / gewijzigd / ongewijzigd zijn
# Dit is handig voor logging en controle.
new_count = 0
changed_count = 0
preview_unchanged_count = 0

for _, src_row in df_klant_source.iterrows():
    klantnr = src_row["klantnr"]
    source_id = src_row["source_id"]

    # Zoek huidig actief record op basis van de nieuwe business key
    current_match = df_klant_dwh[
        (df_klant_dwh["klantnr"] == klantnr) &
        (df_klant_dwh["source_id"] == source_id)
    ]

    if current_match.empty:
        new_count += 1
    else:
        dwh_row = current_match.iloc[0]

        if klant_is_changed(src_row, dwh_row):
            changed_count += 1
        else:
            preview_unchanged_count += 1

logger.info(f"Aantal nieuwe klanten: {new_count}")
logger.info(f"Aantal gewijzigde klanten: {changed_count}")
logger.info(f"Aantal ongewijzigde klanten: {preview_unchanged_count}")


# 6. Echte ETL uitvoeren met SCD Type 2
# Regels:
# - nieuw business key record -> INSERT
# - bestaand business key record + wijziging -> UPDATE oude rij + INSERT nieuwe rij
# - bestaand business key record + geen wijziging -> niets doen

now_ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

inserted_count = 0
updated_count = 0
unchanged_count = 0

logger.info("Start ETL voor Dim_Klant (SCD Type 2)")

for _, src_row in df_klant_source.iterrows():
    klantnr = src_row["klantnr"]
    source_id = src_row["source_id"]

    # Zoek match op business key: klantnr + source_id
    current_match = df_klant_dwh[ (df_klant_dwh["klantnr"] == klantnr) & (df_klant_dwh["source_id"] == source_id) ]

    # Als het een nieuwe rij is -> INSERT
    if current_match.empty:
        dwh_conn.execute("""
            INSERT INTO Dim_Klant (
                klantnr,
                source_id,
                naam,
                adres,
                woonplaats,
                geslacht,
                geboortedatum,
                begintijd,
                eindtijd
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, NULL)
        """, (
            int(src_row["klantnr"]),
            src_row["source_id"],
            src_row["naam"],
            src_row["adres"],
            src_row["woonplaats"],
            src_row["geslacht"],
            src_row["geboortedatum"],
            now_ts
        ))

        inserted_count += 1


    # Bestaande rij -> check of gewijzigd
    else:
        dwh_row = current_match.iloc[0]

        if klant_is_changed(src_row, dwh_row):
            # Stap 1: oude actieve versie afsluiten
            dwh_conn.execute("""
                UPDATE Dim_Klant
                SET eindtijd = ?
                WHERE klant_sk = ?
            """, (
                now_ts,
                int(dwh_row["klant_sk"])
            ))

            # Stap 2: nieuwe versie toevoegen
            dwh_conn.execute("""
                INSERT INTO Dim_Klant (
                    klantnr,
                    source_id,
                    naam,
                    adres,
                    woonplaats,
                    geslacht,
                    geboortedatum,
                    begintijd,
                    eindtijd
                )
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, NULL)
            """, (
                int(src_row["klantnr"]),
                src_row["source_id"],
                src_row["naam"],
                src_row["adres"],
                src_row["woonplaats"],
                src_row["geslacht"],
                src_row["geboortedatum"],
                now_ts
            ))

            updated_count += 1

        else:
            # Geen wijziging -> niets doen
            unchanged_count += 1


# 7. Commit
dwh_conn.commit()

logger.info(
    f"Dim_Klant klaar | inserted={inserted_count}, "
    f"updated_scd2={updated_count}, unchanged={unchanged_count}"
)

# 8. Controle
# Controleer de volledige inhoud van de dimensietabel.

result_df = pd.read_sql_query("""
    SELECT *
    FROM Dim_Klant
    ORDER BY klantnr, source_id, klant_sk
""", dwh_conn)

print(result_df)

Dim_Accessoire (SCD TYPE 1) ROBBERT

In [ ]:
# Wat willen we bereiken?
# We willen Dim_Accessoire in het DWH vullen met SCD Type 1 logica:
# - nieuw accessoire in SDM, nog niet in DWH -> INSERT
# - bestaand accessoire, maar gewijzigd -> UPDATE (overschrijven)
# - bestaand accessoire, niet gewijzigd -> niets doen

# business key = accessoirenr uit SDM
# surrogate key = accessoire_sk in DWH

# 1. Brondata ophalen per bron (horizontale samenvoeging)
# Voor Dim_Accessoire komt de data uit:
# - Accessoire_Verkoop_Accessoire + Accessoire_Verkoop_Leverancier
# - Accessoire_Inkoop_Accessoire + Accessoire_Inkoop_Leverancier

# We JOINEN eerst per bron, omdat accessoire- en leveranciersdata
# in aparte tabellen staan.

query_accessoire_verkoop = """
SELECT
    a.accessoirenr,
    a.naam,
    a.soort,
    l.leveranciernr,
    l.naam AS leveranciernaam,
    l.adres AS leverancieradres,
    l.woonplaats AS leverancierplaats
FROM Accessoire_Verkoop_Accessoire a
JOIN Accessoire_Verkoop_Leverancier l
    ON a.leverancier = l.leveranciernr
"""

query_accessoire_inkoop = """
SELECT
    a.accessoirenr,
    a.naam,
    a.soort,
    l.leveranciernr,
    l.naam AS leveranciernaam,
    l.adres AS leverancieradres,
    l.woonplaats AS leverancierplaats
FROM Accessoire_Inkoop_Accessoire a
JOIN Accessoire_Inkoop_Leverancier l
    ON a.leverancier = l.leveranciernr
"""

df_accessoire_verkoop = pd.read_sql_query(query_accessoire_verkoop, sdm_conn)
df_accessoire_inkoop = pd.read_sql_query(query_accessoire_inkoop, sdm_conn)

logger.info(f"Aantal accessoire-rijen uit verkoopbron: {len(df_accessoire_verkoop)}")
logger.info(f"Aantal accessoire-rijen uit inkoopbron: {len(df_accessoire_inkoop)}")

# 2. Verticale samenvoeging tot één bronset
# We zetten de rijen uit verkoop en inkoop onder elkaar.
# Daarna houden we per business key nog maar één record over.

# Combineer beide bronnen
df_all = pd.concat(
    [
        df_accessoire_verkoop.assign(bron="verkoop"),
        df_accessoire_inkoop.assign(bron="inkoop")
    ],
    ignore_index=True
)

# Groepeer per accessoirenr
df_accessoire_source = []
conflicts = []

for accessoirenr, group in df_all.groupby("accessoirenr"):
    if len(group) == 1:
        # maar 1 bron → gewoon nemen
        df_accessoire_source.append(group.iloc[0])
    else:
        # meerdere bronnen → check of ze gelijk zijn
        unieke = group.drop_duplicates(subset=[
            "naam", "soort",
            "leveranciernr", "leveranciernaam",
            "leverancieradres", "leverancierplaats"
        ])

        if len(unieke) == 1:
            df_accessoire_source.append(unieke.iloc[0])
        else:
            conflicts.append(accessoirenr)
            logger.warning(f"CONFLICT accessoirenr={accessoirenr}")
            print(group)

# maak dataframe
df_accessoire_source = pd.DataFrame(df_accessoire_source)
logger.info(f"Aantal unieke accessoires in source: {len(df_accessoire_source)}")

# 3. Actuele records uit het DWH ophalen
# We vergelijken met de huidige inhoud van Dim_Accessoire.
query_dwh_accessoire = """
SELECT
    accessoire_sk,
    accessoirenr,
    naam,
    soort,
    leveranciernr,
    leveranciernaam,
    leverancieradres,
    leverancierplaats,
    begintijd,
    eindtijd
FROM Dim_Accessoire
WHERE eindtijd IS NULL
"""

df_accessoire_dwh = pd.read_sql_query(query_dwh_accessoire, dwh_conn)

logger.info(f"Aantal actuele accessoires in DWH: {len(df_accessoire_dwh)}")

# 4. Helperfunctie: is het accessoire veranderd?
# Bij SCD Type 1 vergelijken we eerst of de inhoudelijke attributen veranderd zijn.
# Alleen dan voeren we een UPDATE uit.

def accessoire_is_changed(source_row, dwh_row):
    return (
        str(source_row["naam"]) != str(dwh_row["naam"]) or
        str(source_row["soort"]) != str(dwh_row["soort"]) or
        str(source_row["leveranciernr"]) != str(dwh_row["leveranciernr"]) or
        str(source_row["leveranciernaam"]) != str(dwh_row["leveranciernaam"]) or
        str(source_row["leverancieradres"]) != str(dwh_row["leverancieradres"]) or
        str(source_row["leverancierplaats"]) != str(dwh_row["leverancierplaats"])
    )

# 5. Vooraf bepalen hoeveel records nieuw / gewijzigd / ongewijzigd zijn
new_count = 0
changed_count = 0
preview_unchanged_count = 0

for _, src_row in df_accessoire_source.iterrows():
    accessoirenr = src_row["accessoirenr"]

    current_match = df_accessoire_dwh[df_accessoire_dwh["accessoirenr"] == accessoirenr]

    if current_match.empty:
        new_count += 1
    else:
        dwh_row = current_match.iloc[0]

        if accessoire_is_changed(src_row, dwh_row):
            changed_count += 1
        else:
            preview_unchanged_count += 1

logger.info(f"Aantal nieuwe accessoires: {new_count}")
logger.info(f"Aantal gewijzigde accessoires: {changed_count}")
logger.info(f"Aantal ongewijzigde accessoires: {preview_unchanged_count}")

# 6. Echte ETL uitvoeren met SCD Type 1
# - nieuw -> INSERT
# - gewijzigd -> UPDATE
# - ongewijzigd -> niets doen

now_ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

inserted_count = 0
updated_count = 0
unchanged_count = 0

logger.info("Start ETL voor Dim_Accessoire (SCD Type 1)")

for _, src_row in df_accessoire_source.iterrows():
    accessoirenr = src_row["accessoirenr"]

    current_match = df_accessoire_dwh[df_accessoire_dwh["accessoirenr"] == accessoirenr]

    # Nieuw accessoire -> INSERT
    if current_match.empty:
        dwh_conn.execute("""
            INSERT INTO Dim_Accessoire (
                accessoirenr,
                naam,
                soort,
                leveranciernr,
                leveranciernaam,
                leverancieradres,
                leverancierplaats,
                begintijd,
                eindtijd
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, NULL)
        """, (
            int(src_row["accessoirenr"]),
            src_row["naam"],
            src_row["soort"],
            int(src_row["leveranciernr"]),
            src_row["leveranciernaam"],
            src_row["leverancieradres"],
            src_row["leverancierplaats"],
            now_ts
        ))
        inserted_count += 1
    # Bestaand accessoire -> check of gewijzigd
    else:
        dwh_row = current_match.iloc[0]

        if accessoire_is_changed(src_row, dwh_row):
            # Bij SCD Type 1 overschrijven we de bestaande rij.
            dwh_conn.execute("""
                UPDATE Dim_Accessoire
                SET
                    naam = ?,
                    soort = ?,
                    leveranciernr = ?,
                    leveranciernaam = ?,
                    leverancieradres = ?,
                    leverancierplaats = ?
                WHERE accessoire_sk = ?
            """, (
                src_row["naam"],
                src_row["soort"],
                int(src_row["leveranciernr"]),
                src_row["leveranciernaam"],
                src_row["leverancieradres"],
                src_row["leverancierplaats"],
                int(dwh_row["accessoire_sk"])
            ))
            updated_count += 1
        else:
            unchanged_count += 1

dwh_conn.commit()

logger.info(
    f"Dim_Accessoire klaar | inserted={inserted_count}, "
    f"updated_scd1={updated_count}, unchanged={unchanged_count}"
)

# 7. Controle
result_df = pd.read_sql_query("""
    SELECT *
    FROM Dim_Accessoire
    ORDER BY accessoirenr, accessoire_sk
""", dwh_conn)

print(result_df)

Dim_Datum ROBBERT

In [ ]:
# Wat willen we bereiken?
# We willen Dim_Datum in het DWH vullen.
# Voor Dim_Datum geldt:
# - nieuwe datum in SDM, nog niet in DWH -> INSERT
# - bestaande datum in DWH -> niets doen
# Voor Dim_Datum gebruiken we geen SCD Type 1 of Type 2,
# omdat datumwaarden zelf niet wijzigen.

# business key = datum
# surrogate key = datum_sk in DWH

# 1. Brondata ophalen uit de relevante feitbronnen=
# We halen de datums op uit:
# - Fiets_Verkoop
# - Accessoire_Verkoop
# - Onderhoud
# Voor Fct_Inkoop hebben we in het SDM geen echte datumkolom,
# maar maand + jaar. Daarom gebruiken we hier alleen de tabellen
# die een volledige datum bevatten.

query_fiets_verkoop_datum = """
SELECT datum
FROM Fiets_Verkoop
"""

query_accessoire_verkoop_datum = """
SELECT datum
FROM Accessoire_Verkoop
"""

query_onderhoud_datum = """
SELECT datum
FROM Onderhoud
"""

df_fiets_verkoop_datum = pd.read_sql_query(query_fiets_verkoop_datum, sdm_conn)
df_accessoire_verkoop_datum = pd.read_sql_query(query_accessoire_verkoop_datum, sdm_conn)
df_onderhoud_datum = pd.read_sql_query(query_onderhoud_datum, sdm_conn)

logger.info(f"Aantal datum-rijen uit Fiets_Verkoop: {len(df_fiets_verkoop_datum)}")
logger.info(f"Aantal datum-rijen uit Accessoire_Verkoop: {len(df_accessoire_verkoop_datum)}")
logger.info(f"Aantal datum-rijen uit Onderhoud: {len(df_onderhoud_datum)}")

# 2. Verticale samenvoeging tot één bronset
# We zetten alle datumrijen onder elkaar.
# Daarna houden we alleen unieke datums over.

df_datum_source = pd.concat(
    [df_fiets_verkoop_datum, df_accessoire_verkoop_datum, df_onderhoud_datum],
    ignore_index=True
)

df_datum_source = df_datum_source.drop_duplicates(subset=["datum"]).reset_index(drop=True)

logger.info(f"Aantal unieke datums in source: {len(df_datum_source)}")

# 3. Datumonderdelen afleiden
# We zetten de datumkolom om naar datetime,
# zodat we year, month en day kunnen afleiden.

df_datum_source["datum"] = pd.to_datetime(df_datum_source["datum"])
df_datum_source["year"] = df_datum_source["datum"].dt.year
df_datum_source["month"] = df_datum_source["datum"].dt.month
df_datum_source["day"] = df_datum_source["datum"].dt.day

# Voor opslag in SQLite zetten we datum weer om naar YYYY-MM-DD string
df_datum_source["datum"] = df_datum_source["datum"].dt.strftime("%Y-%m-%d")

# 4. Bestaande datums uit DWH ophalen
query_dwh_datum = """
SELECT
    datum_sk,
    datum,
    year,
    month,
    day
FROM Dim_Datum
"""

df_datum_dwh = pd.read_sql_query(query_dwh_datum, dwh_conn)

logger.info(f"Aantal datums in DWH: {len(df_datum_dwh)}")

# 5. Bepalen welke datums nieuw zijn
new_count = 0
existing_count = 0

for _, src_row in df_datum_source.iterrows():
    datum = src_row["datum"]

    current_match = df_datum_dwh[df_datum_dwh["datum"] == datum]

    if current_match.empty:
        new_count += 1
    else:
        existing_count += 1

logger.info(f"Aantal nieuwe datums: {new_count}")
logger.info(f"Aantal bestaande datums: {existing_count}")

# 6. Echte ETL uitvoeren
# Alleen nieuwe datums worden toegevoegd.

inserted_count = 0
logger.info("Start ETL voor Dim_Datum")

for _, src_row in df_datum_source.iterrows():
    datum = src_row["datum"]

    current_match = df_datum_dwh[df_datum_dwh["datum"] == datum]

    if current_match.empty:
        dwh_conn.execute("""
            INSERT INTO Dim_Datum (
                datum,
                year,
                month,
                day
            )
            VALUES (?, ?, ?, ?)
        """, (
            src_row["datum"],
            int(src_row["year"]),
            int(src_row["month"]),
            int(src_row["day"])
        ))
        inserted_count += 1

dwh_conn.commit()
logger.info(f"Dim_Datum klaar | inserted={inserted_count}")

# 7. Controle

result_df = pd.read_sql_query("""
    SELECT *
    FROM Dim_Datum
    ORDER BY datum
""", dwh_conn)

print(result_df)

Dim_Fiets (SCD Type 2) MEES

In [ ]:
logger.info("Start ETL Dim_Fiets (SCD Type 2 met business key fietsnr + source_id)")

# =========================================================
# 1. BRONDATA OPHALEN
# =========================================================

df_inkoop = pd.read_sql_query("""
SELECT 
    f.fietsnr, f.soort, f.merk, f.type, f.kleur,
    fab.fabrikantnr, fab.naam, fab.adres, fab.plaats
FROM Fiets_Inkoop_Fiets f
JOIN Fiets_Inkoop_Fabrikant fab ON f.fabrikant = fab.fabrikantnr
""", sdm_conn)

df_verkoop = pd.read_sql_query("""
SELECT 
    f.fietsnr, f.soort, f.merk, f.type, f.kleur,
    fab.fabrikantnr, fab.naam, fab.adres, fab.plaats
FROM Fiets_Verkoop_Fiets f
JOIN Fiets_Verkoop_Fabrikant fab ON f.fabrikant = fab.fabrikantnr
""", sdm_conn)

df_onderhoud = pd.read_sql_query("""
SELECT 
    f.fietsnr, f.soort, f.merk, f.type, f.kleur,
    fab.fabrikantnr, fab.naam, fab.adres, fab.plaats
FROM Onderhoud_Fiets f
JOIN Onderhoud_Fabrikant fab ON f.fabrikant = fab.fabrikantnr
""", sdm_conn)

# source_id toevoegen
df_inkoop["source_id"] = "Fiets_Inkoop"
df_verkoop["source_id"] = "Fiets_Verkoop"
df_onderhoud["source_id"] = "Onderhoud"

# samenvoegen
df_fiets_source = pd.concat(
    [df_inkoop, df_verkoop, df_onderhoud],
    ignore_index=True
)

logger.info(f"Aantal bronrecords: {len(df_fiets_source)}")

# =========================================================
# 2. DUBBELE CHECK (BINNEN ZELFDE BUSINESS KEY)
# =========================================================

duplicate_bk = df_fiets_source.duplicated(
    subset=["fietsnr", "source_id"], keep=False
)

if duplicate_bk.any():
    logger.warning("Dubbele records binnen dezelfde business key gevonden")

# =========================================================
# 3. ACTUELE DWH DATA
# =========================================================

df_dwh = pd.read_sql_query("""
SELECT *
FROM Dim_Fiets
WHERE eindtijd IS NULL
""", dwh_conn)

logger.info(f"Aantal actuele fietsen in DWH: {len(df_dwh)}")

# =========================================================
# 4. WIJZIGINGSCHECK
# =========================================================

def fiets_is_changed(src, dwh):
    return (
        str(src["soort"]) != str(dwh["soort"]) or
        str(src["merk"]) != str(dwh["merk"]) or
        str(src["type"]) != str(dwh["type"]) or
        str(src["kleur"]) != str(dwh["kleur"]) or
        str(src["fabrikantnr"]) != str(dwh["fabrikantnr"]) or
        str(src["naam"]) != str(dwh["fabrikantnaam"]) or
        str(src["adres"]) != str(dwh["fabrikantadres"]) or
        str(src["plaats"]) != str(dwh["fabrikantplaats"])
    )

# =========================================================
# 5. PREVIEW
# =========================================================

new_count = 0
changed_count = 0
unchanged_count = 0

for _, src in df_fiets_source.iterrows():
    match = df_dwh[
        (df_dwh["fietsnr"] == src["fietsnr"]) &
        (df_dwh["source_id"] == src["source_id"])
    ]

    if match.empty:
        new_count += 1
    else:
        if fiets_is_changed(src, match.iloc[0]):
            changed_count += 1
        else:
            unchanged_count += 1

logger.info(f"Preview | new={new_count}, changed={changed_count}, unchanged={unchanged_count}")

# =========================================================
# 6. ETL (SCD TYPE 2)
# =========================================================

now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

inserted = 0
updated = 0
unchanged = 0

for _, src in df_fiets_source.iterrows():

    fietsnr = int(src["fietsnr"])
    source_id = src["source_id"]

    match = df_dwh[
        (df_dwh["fietsnr"] == fietsnr) &
        (df_dwh["source_id"] == source_id)
    ]

    # NIEUW
    if match.empty:
        dwh_conn.execute("""
        INSERT INTO Dim_Fiets (
            fietsnr, soort, merk, type, kleur,
            fabrikantnr, fabrikantnaam, fabrikantadres, fabrikantplaats,
            begintijd, eindtijd, source_id
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, NULL, ?)
        """, (
            fietsnr,
            src["soort"],
            src["merk"],
            src["type"],
            src["kleur"],
            int(src["fabrikantnr"]),
            src["naam"],
            src["adres"],
            src["plaats"],
            now,
            source_id
        ))

        inserted += 1

    # BESTAAND
    else:
        dwh_row = match.iloc[0]

        if fiets_is_changed(src, dwh_row):

            # afsluiten
            dwh_conn.execute("""
            UPDATE Dim_Fiets
            SET eindtijd = ?
            WHERE fiets_sk = ?
            """, (now, int(dwh_row["fiets_sk"])))

            # nieuwe versie
            dwh_conn.execute("""
            INSERT INTO Dim_Fiets (
                fietsnr, soort, merk, type, kleur,
                fabrikantnr, fabrikantnaam, fabrikantadres, fabrikantplaats,
                begintijd, eindtijd, source_id
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, NULL, ?)
            """, (
                fietsnr,
                src["soort"],
                src["merk"],
                src["type"],
                src["kleur"],
                int(src["fabrikantnr"]),
                src["naam"],
                src["adres"],
                src["plaats"],
                now,
                source_id
            ))

            updated += 1
        else:
            unchanged += 1

# =========================================================
# 7. COMMIT
# =========================================================

dwh_conn.commit()

logger.success(
    f"Dim_Fiets klaar | inserted={inserted}, updated={updated}, unchanged={unchanged}"
)

# =========================================================
# 8. CONTROLE
# =========================================================

result = pd.read_sql_query("""
SELECT *
FROM Dim_Fiets
ORDER BY fietsnr, source_id, fiets_sk
""", dwh_conn)

print(result)

Dim_Monteur (SCD Type 1) MEES

In [ ]:
logger.info("Ophalen Monteur uit SDM")

sdm_cursor.execute("""
SELECT DISTINCT
    m.monteurnr,
    m.naam,
    m.woonplaats,
    m.uurloon,
    f.filiaalnr,
    f.naam,
    f.adres,
    f.provincie
FROM Accessoire_Verkoop_Monteur m
JOIN Accessoire_Verkoop_Filiaal f ON m.filiaal = f.filiaalnr

UNION ALL
                                   
SELECT DISTINCT
    m.monteurnr,
    m.naam,
    m.woonplaats,
    m.uurloon,
    f.filiaalnr,
    f.naam,
    f.adres,
    f.provincie
FROM Fiets_Verkoop_Monteur m
JOIN Fiets_Verkoop_Filiaal f ON m.filiaal = f.filiaalnr

UNION ALL

SELECT DISTINCT
    m.monteurnr,
    m.naam,
    m.woonplaats,
    m.uurloon,
    f.filiaalnr,
    f.naam,
    f.adres,
    f.provincie
FROM Onderhoud_Monteur m
JOIN Onderhoud_Filiaal f ON m.filiaal = f.filiaalnr
""")

source_data = sdm_cursor.fetchall()

dwh_cursor.execute("""
SELECT 
    monteurnr,
    naam,
    woonplaats,
    uurloon,
    filiaalnr,
    filiaalnaam,
    filiaaladres,
    filiaalprovincie,
    loonklasse,
    monteur_sk
FROM Dim_Monteur
WHERE eindtijd IS NULL
""")

dwh_data = {row[0]: row for row in dwh_cursor.fetchall()}
logger.info("Data opgehaald uit SDM en DWH.")

processed_keys = set()

for row in source_data:
    (
        monteurnr,
        naam,
        woonplaats,
        uurloon,
        filiaalnr,
        filiaalnaam,
        filiaaladres,
        filiaalprovincie
    ) = row

    if monteurnr in processed_keys:
        continue
    processed_keys.add(monteurnr)

    now = datetime.now()
    loonklasse = "hoog" if uurloon >= 20 else "laag"

    if monteurnr not in dwh_data:
        logger.info(f"Nieuwe monteur: {monteurnr}")

        dwh_cursor.execute("""
        INSERT INTO Dim_Monteur (
            monteurnr,
            naam,
            woonplaats,
            uurloon,
            filiaalnr,
            filiaalnaam,
            filiaaladres,
            filiaalprovincie,
            begintijd,
            eindtijd,
            loonklasse
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, NULL, ?)
        """, (
            monteurnr,
            naam,
            woonplaats,
            uurloon,
            filiaalnr,
            filiaalnaam,
            filiaaladres,
            filiaalprovincie,
            now,
            loonklasse
        ))

    else:
        (
            _,
            d_naam,
            d_woonplaats,
            d_uurloon,
            d_filiaalnr,
            d_filiaalnaam,
            d_filiaaladres,
            d_filiaalprovincie,
            d_loonklasse,
            monteur_sk
        ) = dwh_data[monteurnr]

        if (
            naam,
            woonplaats,
            uurloon,
            filiaalnr,
            filiaalnaam,
            filiaaladres,
            filiaalprovincie,
            loonklasse
        ) != (
            d_naam,
            d_woonplaats,
            d_uurloon,
            d_filiaalnr,
            d_filiaalnaam,
            d_filiaaladres,
            d_filiaalprovincie,
            d_loonklasse
        ):

            logger.info(f"Update monteur (Type 1): {monteurnr}")

            dwh_cursor.execute("""
            UPDATE Dim_Monteur
            SET naam = ?,
                woonplaats = ?,
                uurloon = ?,
                filiaalnr = ?,
                filiaalnaam = ?,
                filiaaladres = ?,
                filiaalprovincie = ?,
                loonklasse = ?
            WHERE monteur_sk = ?
            """, (
                naam,
                woonplaats,
                uurloon,
                filiaalnr,
                filiaalnaam,
                filiaaladres,
                filiaalprovincie,
                loonklasse,
                monteur_sk
            ))

dwh_conn.commit()
logger.info("Monteur dimensie geupdate in DWH.")

Fct_Onderhoud MEES

In [ ]:
logger.info("Ophalen Onderhoud uit SDM")

sdm_cursor.execute("""
SELECT 
    o.onderhoudnr,
    o.datum,
    o.starttijd,
    o.eindtijd,
    o.fiets,
    o.monteur
FROM Onderhoud o
""")

source_data = sdm_cursor.fetchall()

dwh_cursor.execute("SELECT onderhoudnr FROM Fct_Onderhoud")
bestaande_ids = {row[0] for row in dwh_cursor.fetchall()}

def get_or_create_datum_sk(datum):
    dwh_cursor.execute("SELECT datum_sk FROM Dim_Datum WHERE datum = ?", (datum,))
    if r := dwh_cursor.fetchone():
        return r[0]

    from datetime import datetime
    d = datetime.strptime(datum, "%Y-%m-%d")

    dwh_cursor.execute("""
    INSERT INTO Dim_Datum (datum, year, month, day)
    VALUES (?, ?, ?, ?)
    """, (datum, d.year, d.month, d.day))

    return dwh_cursor.lastrowid


def get_fiets_sk(fietsnr):
    if fietsnr is None:
        return None
    dwh_cursor.execute("""
    SELECT fiets_sk FROM Dim_Fiets
    WHERE fietsnr = ? AND eindtijd IS NULL
    """, (fietsnr,))
    r = dwh_cursor.fetchone()
    return r[0] if r else None


def get_monteur_sk(monteurnr):
    dwh_cursor.execute("""
    SELECT monteur_sk FROM Dim_Monteur
    WHERE monteurnr = ? AND eindtijd IS NULL
    """, (monteurnr,))
    result = dwh_cursor.fetchone()
    return result[0] if result else None

def get_monteur_uurloon(monteurnr):
    dwh_cursor.execute("""
    SELECT uurloon FROM Dim_Monteur
    WHERE monteurnr = ? AND eindtijd IS NULL
    """, (monteurnr,))
    result = dwh_cursor.fetchone()
    return result[0] if result else None

def parse_tijd(t):
    if t is None:
        return None

    if "." in t:
        hoofd, fractie = t.split(".")
        fractie = fractie[:6]  # max 6 digits
        t = f"{hoofd}.{fractie}"

    for fmt in ("%H:%M:%S.%f", "%H:%M:%S"):
        try:
            return datetime.strptime(t, fmt)
        except ValueError:
            continue

    raise ValueError(f"Ongeldig tijdformaat: {t}")

def bereken_uren(starttijd, eindtijd):
    start = parse_tijd(starttijd)
    eind = parse_tijd(eindtijd)

    delta = eind - start
    return delta.total_seconds() / 3600



logger.info("Start ETL Fct_Onderhoud")

for row in source_data:
    onderhoudnr, datum, starttijd, eindtijd, fietsnr, monteurnr = row

    # skip bestaande
    if onderhoudnr in bestaande_ids:
        continue

    datum_sk = get_or_create_datum_sk(datum)
    fiets_sk = get_fiets_sk(fietsnr)
    monteur_sk = get_monteur_sk(monteurnr)

    uren = bereken_uren(starttijd, eindtijd) if starttijd and eindtijd else None
    uurloon = get_monteur_uurloon(monteurnr)
    arbeidskosten = uren * uurloon if uren and uurloon else None

    # Check of alle keys bestaan
    if not all([datum_sk, fiets_sk, monteur_sk]):
        logger.warning(f"SK ontbreekt voor onderhoud {onderhoudnr}")
        continue

    # Insert fact
    dwh_cursor.execute("""
    INSERT INTO Fct_Onderhoud (
        onderhoudnr,
        datum_sk,
        starttijd,
        eindtijd,
        fiets_sk,
        monteur_sk,
        arbeidskosten
    )
    VALUES (?, ?, ?, ?, ?, ?, ?)
    """, (onderhoudnr, datum_sk, starttijd, eindtijd, fiets_sk, monteur_sk, arbeidskosten))

dwh_conn.commit()
logger.success("Fct_Onderhoud load klaar")

Fct_Verkoop ROB

In [ ]:
logger.info("Ophalen Verkoop uit SDM")

sdm_cursor.execute("""
SELECT 
    fv.fiets_verkoopnr AS verkoopnr,
    fv.datum,
    fv.aantal,
    fv.verkoopprijs,
    ff.standaardprijs,
    fv.klant,
    fv.monteur,
    fv.fiets,
    NULL as accessoire
FROM Fiets_Verkoop fv
JOIN Fiets_Verkoop_Fiets ff ON fv.fiets = ff.fietsnr

UNION ALL

SELECT 
    av.accessoire_verkoopnr AS verkoopnr,
    av.datum,
    av.aantal,
    av.verkoopprijs,
    aa.standaardprijs,
    av.klant,
    av.monteur,
    NULL as fiets,
    av.accessoire
FROM Accessoire_Verkoop av
JOIN Accessoire_Verkoop_Accessoire aa ON av.accessoire = aa.accessoirenr
""")

source_data = sdm_cursor.fetchall()

# incremental check (verbeterde key)
dwh_cursor.execute("""
SELECT verkoop_type, datum_sk, aantal, klant_sk, totaalprijs FROM Fct_Verkoop
""")
bestaande_records = set(dwh_cursor.fetchall())


# --- DIM LOOKUPS ---
def get_fiets_sk(fietsnr):
    if fietsnr is None:
        return None
    dwh_cursor.execute("""
    SELECT fiets_sk FROM Dim_Fiets
    WHERE fietsnr = ? AND eindtijd IS NULL
    """, (fietsnr,))
    r = dwh_cursor.fetchone()
    return r[0] if r else None


def get_accessoire_sk(accessoirenr):
    if accessoirenr is None:
        return None
    dwh_cursor.execute("""
    SELECT accessoire_sk FROM Dim_Accessoire
    WHERE accessoirenr = ? AND eindtijd IS NULL
    """, (accessoirenr,))
    r = dwh_cursor.fetchone()
    return r[0] if r else None


def get_klant_sk(klantnr):
    dwh_cursor.execute("""
    SELECT klant_sk FROM Dim_Klant
    WHERE klantnr = ? AND eindtijd IS NULL
    """, (klantnr,))
    r = dwh_cursor.fetchone()
    return r[0] if r else None


def get_monteur_sk(monteurnr):
    dwh_cursor.execute("""
    SELECT monteur_sk FROM Dim_Monteur
    WHERE monteurnr = ? AND eindtijd IS NULL
    """, (monteurnr,))
    r = dwh_cursor.fetchone()
    return r[0] if r else None


logger.info("Start ETL Fct_Verkoop")

inserted = 0
skipped = 0
errors = 0

for row in source_data:
    try:
        verkoopnr, datum, aantal, verkoopprijs, standaardprijs, klantnr, monteurnr, fietsnr, accessoirenr = row

        datum_sk = get_or_create_datum_sk(datum)
        fiets_sk = get_fiets_sk(fietsnr)
        accessoire_sk = get_accessoire_sk(accessoirenr)
        klant_sk = get_klant_sk(klantnr)
        monteur_sk = get_monteur_sk(monteurnr)

        # bepaal type
        if fiets_sk is not None:
            verkoop_type = "fiets"
        elif accessoire_sk is not None:
            verkoop_type = "accessoire"
        else:
            logger.warning(f"Ongeldige verkoop (geen match): {verkoopnr}")
            continue

        # validatie
        if fiets_sk is not None and accessoire_sk is not None:
            logger.warning(f"Dubbele koppeling bij verkoop {verkoopnr}")
            continue

        aantal = int(aantal)
        totaalprijs = float(verkoopprijs)
        standaardprijs = float(standaardprijs)
        omzet = totaalprijs

        # VERBETERDE INCREMENTAL KEY
        record_key = (
            verkoop_type,
            datum_sk,
            aantal,
            klant_sk,
            totaalprijs
        )

        if record_key in bestaande_records:
            skipped += 1
            continue

        dwh_cursor.execute("""
        INSERT INTO Fct_Verkoop (
            verkoop_type,
            datum_sk,
            aantal,
            totaalprijs,
            standaardprijs,
            klant_sk,
            fiets_sk,
            accessoire_sk,
            monteur_sk,
            omzet
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            verkoop_type,
            datum_sk,
            aantal,
            totaalprijs,
            standaardprijs,
            klant_sk,
            fiets_sk,
            accessoire_sk,
            monteur_sk,
            omzet
        ))

        bestaande_records.add(record_key)
        inserted += 1

    except Exception as e:
        logger.error(f"Fout bij row {row}: {e}")
        errors += 1


dwh_conn.commit()

logger.info(f"Fct_Verkoop klaar inserted={inserted}, skipped={skipped}, errors={errors}")

Fct_Inkoop MEES

In [ ]:
logger.info("Ophalen Inkoop uit SDM") 

sdm_cursor.execute("""
SELECT 
    i.inkoopnr,
    DATE(i.inkoopjaar || '-' || printf('%02d', i.inkoopmaand) || '-01') AS datum,
    i.aantal,
    f.inkoopprijs,
    i.fiets,
    NULL as accessoire
FROM Fiets_Inkoop i
JOIN Fiets_Inkoop_Fiets f ON i.fiets = f.fietsnr

UNION ALL

SELECT 
    i.inkoopnr,
    DATE(i.inkoopjaar || '-' || printf('%02d', i.inkoopmaand) || '-01') AS datum,
    i.aantal,
    a.inkoopprijs,
    NULL as fiets,
    i.accessoire
FROM Accessoire_Inkoop i
JOIN Accessoire_Inkoop_Accessoire a ON i.accessoire = a.accessoirenr
""")

source_data = sdm_cursor.fetchall()

# LET OP: we gebruiken nu de SOURCE inkoopnr alleen voor incremental check
dwh_cursor.execute("SELECT inkoop_type, datum_sk, aantal FROM Fct_Inkoop")
bestaande_records = set(dwh_cursor.fetchall())


def get_fiets_sk(fietsnr):
    if fietsnr is None:
        return None
    dwh_cursor.execute("""
    SELECT fiets_sk FROM Dim_Fiets
    WHERE fietsnr = ? AND eindtijd IS NULL
    """, (fietsnr,))
    r = dwh_cursor.fetchone()
    return r[0] if r else None


def get_accessoire_sk(accessoirenr):
    if accessoirenr is None:
        return None
    dwh_cursor.execute("""
    SELECT accessoire_sk FROM Dim_Accessoire
    WHERE accessoirenr = ? AND eindtijd IS NULL
    """, (accessoirenr,))
    r = dwh_cursor.fetchone()
    return r[0] if r else None


logger.info("Start ETL Fct_Inkoop")

for row in source_data:
    source_inkoopnr, datum, aantal, inkoopprijs, fietsnr, accessoirenr = row

    datum_sk = get_or_create_datum_sk(datum)
    fiets_sk = get_fiets_sk(fietsnr)
    accessoire_sk = get_accessoire_sk(accessoirenr)

    # bepaal type
    if fiets_sk is not None:
        inkoop_type = "fiets"
    elif accessoire_sk is not None:
        inkoop_type = "accessoire"
    else:
        logger.warning(f"Ongeldige inkoop (geen match): {source_inkoopnr}")
        continue

    # validatie (extra check)
    if fiets_sk is not None and accessoire_sk is not None:
        logger.warning(f"Dubbele koppeling bij {source_inkoopnr}")
        continue

    totaalprijs = aantal * inkoopprijs

    # incremental check (zonder auto-id)
    record_key = (inkoop_type, datum_sk, aantal)
    if record_key in bestaande_records:
        continue

    dwh_cursor.execute("""
    INSERT INTO Fct_Inkoop (
        inkoop_type,
        datum_sk,
        aantal,
        totaalprijs,
        inkoopprijs,
        fiets_sk,
        accessoire_sk
    )
    VALUES (?, ?, ?, ?, ?, ?, ?)
    """, (
        inkoop_type,
        datum_sk,
        aantal,
        totaalprijs,
        inkoopprijs,
        fiets_sk,
        accessoire_sk
    ))

dwh_conn.commit()
logger.success("Fct_Inkoop load klaar")